# Worked solutions

This notebook is identical to the student version except that every 🔵 `# TODO` has been filled in, **with commentary on why the answer is what it is** rather than just the code. Read the comments — the reasoning is the point, not the syntax.

Everything else, including the ✏️ YOUR TURN cells, is unchanged: those have no single right answer.


# Module D — Confounding

## Is the model learning the disease, or learning who was recruited?

### What you will be able to do by the end

1. explain what a confounder is, and recognise one in a real dataset
2. show that a model with a good score can be learning something you did not intend
3. use stratification, adjustment and matching to take a confounder apart
4. demonstrate **Simpson's paradox** on real data — a trend that reverses when you split the groups
5. explain why 'the model is accurate' and 'the model is useful' are different claims

### The data

**Real data, all of it.** This is **OASIS-1**: 416 real people scanned once each, with age, sex, education, socioeconomic status, a cognitive score, a clinical dementia rating, and the volumetric measures from their MRI. It is ideal for this module because of a design choice the OASIS team made: the cohort deliberately spans **ages 18 to 96**, and everybody under about 60 is by construction healthy. Age is therefore entangled with diagnosis in the most extreme way possible — and that is true, to a lesser degree, of nearly every dementia cohort ever assembled.

### How to work through this notebook

Run the cells in order, top to bottom. The notebook is split into four sections:

| | Section | What happens |
|---|---|---|
| 1 | **Understand the data** | Meet every column and every person in the table |
| 2 | **Quality control** | Find the flaws before they fool you |
| 3 | **Build models** | Start from something trivial, then climb |
| 4 | **Read the results** | Turn numbers into a clinical judgement |

Look out for these markers:

- ✏️ **YOUR TURN** — change the value shown, re-run the cell, watch the figure change. Everyone does these.
- 🟢 run and read · 🔵 write a little code · ⚫ take home
- 🧠 a question to think about; the answer is hidden underneath, so try first

**In a hurry?** Run sections 1 and 2 quickly and spend your time in section 3 — this module's whole point lives there.

---

*Teaching material. Nothing here is a diagnostic tool, and no result in this notebook is clinical evidence.*


In [ ]:
# Run me first. This finds the project folder, loads the shared helpers,
# and prints exactly where this module's data came from.
from pathlib import Path
import sys
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plots
from data import load_data, load_extra, provenance
from models import split_data, train_model, evaluate, compare_models, sweep_parameter, MODEL_CHOICES

pd.set_option('display.width', 160)
print(provenance('D'))


---
# 1 · Understand the data

One row is one person, scanned once.


### 1.1 The table

| Column | Meaning |
|---|---|
| `age` | 18 to 96. Look at that range — it is the whole module. |
| `sex`, `handedness` | |
| `education_code` | Coded 1–5, higher = more education. A proxy for **cognitive reserve**. |
| `ses` | Socioeconomic status, 1 (highest) to 5 (lowest). Frequently missing. |
| `mmse` | Mini-Mental State Examination, 0–30. Missing for the younger participants, who were not tested. |
| `cdr` | Clinical Dementia Rating: 0 = none, 0.5 = very mild, 1 = mild, 2 = moderate. Missing for the young. |
| `etiv_mm3` | Skull cavity volume — head size, which does not change with disease. |
| `nwbv` | Normalised whole-brain volume: the fraction still filled with brain. **The atrophy measure.** |
| `asf` | Atlas scaling factor, essentially 1/eTIV. |
| `impaired` | Our label: `cdr > 0`. Missing wherever `cdr` is. |


In [ ]:
df = load_data('D')
print(f'{len(df)} people, aged {df.age.min():.0f} to {df.age.max():.0f}.')
print(f'{df.impaired.notna().sum()} of them have a clinical dementia rating; the rest were never assessed.\n')
df.head()


### 1.2 The shape of the problem

**Predict before you run:** in this cohort, how much do the ages of the impaired and unimpaired groups overlap?


In [ ]:
labelled = df.dropna(subset=['impaired']).copy()
labelled['status'] = np.where(labelled['impaired'] == 1, 'impaired', 'unimpaired')

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist([df.loc[df.impaired.isna(), 'age'],
         labelled.loc[labelled.impaired == 0, 'age'],
         labelled.loc[labelled.impaired == 1, 'age']],
        bins=20, stacked=True, color=['#cccccc', '#2c6fbb', '#e08214'],
        label=['never assessed (young)', 'unimpaired', 'impaired'])
ax.set_xlabel('age (years)'); ax.set_ylabel('number of people')
ax.set_title('The cohort by age. Nobody under ~60 is in the orange group — by design.')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print(labelled.groupby('status')[['age', 'nwbv', 'education_code', 'mmse']].mean().round(3))


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** read the mean ages of the two groups above and predict what a model given only `age` would score.
- 🔵 **If you want to write code:** plot `nwbv` against `age` for the unimpaired group only, and fit a straight line through it. That slope is *normal ageing* — everything a model attributes to disease should be measured against it.
- ⚫ **Take home:** look up how 'brain age' models work: they predict chronological age from a scan, and the *residual* (brain older than it should be) becomes the disease marker. It is confounder-adjustment turned into a method.


---
# 2 · Quality control

Two structural problems, both of which are about *who is in the table*, not about *what the numbers say*.


### 2.1 Missingness that is structural, not accidental

`cdr` and `mmse` are missing for a large block of participants — not at random, but because the young ones were never given a dementia assessment. `ses` is missing for a different reason again.

Dropping the rows without a `cdr` is the right call here, but notice what it does: it removes the entire young half of the cohort, which is exactly the half that would have shown you what normal looks like.


In [ ]:
plots.plot_missingness(df, title='Missing values across all 416 participants')
plt.show()

by_age = df.assign(no_cdr=df['cdr'].isna(),
                   band=pd.cut(df['age'], [17, 40, 60, 75, 100], labels=['18-40', '40-60', '60-75', '75+']))
rates = by_age.groupby('band', observed=True)['no_cdr'].mean() * 100
plots.plot_score_comparison(rates.index.astype(str).tolist(), rates.tolist(),
                            colours=['#e08214'] * len(rates),
                            title='Percentage never given a dementia rating, by age band',
                            ylabel='percent')
plt.show()


### 2.2 ✏️ Your turn — the age–diagnosis entanglement

Restrict the cohort to a narrow age window and watch what happens to the apparent effect of brain volume. Inside a narrow window, age can no longer explain anything — whatever survives is real.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Change the age window and re-run. Try:
#     (60, 100)  -> the full older cohort, age varies a lot
#     (70, 80)   -> a narrow band; age is nearly constant
#     (75, 85)
#   Watch the 'age-only AUROC' bar collapse as the window narrows.
# ==========================================================================
AGE_MIN = 60
AGE_MAX = 100

window = labelled[(labelled.age >= AGE_MIN) & (labelled.age <= AGE_MAX)]
y_window = window['impaired'].astype(int)
print(f'{len(window)} people aged {AGE_MIN}-{AGE_MAX}; {y_window.mean():.0%} impaired.\n')

sets = {'age only': ['age'],
        'brain volume only': ['nwbv'],
        'both': ['age', 'nwbv']}
scores = {}
for name, columns in sets.items():
    X_tr, X_te, y_tr, y_te = split_data(window[columns], y_window)
    scores[name] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

plots.plot_score_comparison(list(scores), list(scores.values()), reference=0.5,
                            colours=['#e08214', '#2c6fbb', '#5aa469'],
                            title=f'Ages {AGE_MIN}-{AGE_MAX}: what still works when age cannot help?',
                            ylabel='AUROC')
plt.show()


### 2.3 QC verdict

**Usable, and unusually honest about its own limitation.** We keep only the 235 participants with a clinical rating, we accept that `ses` is missing for a substantial minority, and we go into section 3 knowing that age and diagnosis are almost the same variable in this cohort.

*(**Express path:** you can start from section 3 — run its catch-up cell first and everything below stands alone.)*


---
# 3 · Build models — and then take them apart

This module inverts the usual goal. **We are not trying to get the highest score.** We are trying to find out what a good score is made of.


### 🚏 Taking the Express path? Run this one cell first

It rebuilds everything sections 3 and 4 need, so you can start here without having run sections 1 and 2 yourself. **If you did run them, run this anyway** — it just redefines the same things and costs a second.


In [ ]:
# Express catch-up: safe to run whether or not you did sections 1 and 2.
df = load_data('D')
labelled = df.dropna(subset=['impaired']).copy()
labelled['status'] = np.where(labelled['impaired'] == 1, 'impaired', 'unimpaired')
labelled['band'] = pd.cut(labelled['age'], [59, 70, 78, 100], labels=['60-70', '70-78', '78+'])
print(f'{len(labelled)} participants with a clinical dementia rating. Ready for section 3.')


### 3.1 The naive model

Throw everything in. This is what most people do first, and it is not wrong — it is just not yet an answer.


In [ ]:
features = ['age', 'sex', 'education_code', 'ses', 'etiv_mm3', 'nwbv', 'asf']
X = labelled[features]
y = labelled['impaired'].astype(int)
X_train, X_test, y_train, y_test = split_data(X, y)

naive = train_model('logistic', X_train, y_train)
naive_metrics = evaluate(naive, X_test, y_test)
probability = naive.predict_proba(X_test)[:, 1]

plots.plot_roc_pr(y_test, probability, title='Everything-in model — looks respectable')
plt.show()
for name, value in naive_metrics.items():
    print(f'  {name:<20s} {value:.3f}')


### 3.2 Ablation — take one column away at a time

The cheapest way to find out what a model is standing on: remove each feature, refit, and see how far the score falls. A feature whose removal costs nothing was contributing nothing.


In [ ]:
full_score = evaluate(train_model('logistic', X_train, y_train), X_test, y_test)['auroc']
drops = {}
for feature in features:
    remaining = [name for name in features if name != feature]
    X_tr, X_te, y_tr, y_te = split_data(labelled[remaining], y)
    drops[feature] = full_score - evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

plots.plot_importance(list(drops), list(drops.values()),
                      title='How much AUROC is lost when each feature is removed',
                      xlabel='drop in AUROC when this column is deleted')
plt.show()
print(f'Full model AUROC: {full_score:.3f}')


### 3.3 The uncomfortable experiment — predict age instead

If our features are really measuring *disease*, they should be much better at predicting disease than at predicting how old somebody is. Let's check that directly: same features, but the target is now "is this person over 75?".


In [ ]:
brain_only = ['nwbv', 'etiv_mm3', 'asf']

X_tr, X_te, y_tr, y_te = split_data(labelled[brain_only], y)
disease_score = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

old = (labelled['age'] > 75).astype(int)
X_tr, X_te, y_tr, y_te = split_data(labelled[brain_only], old)
age_score = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

plots.plot_score_comparison(['predicting\nimpairment', 'predicting\nbeing over 75'],
                            [disease_score, age_score], reference=0.5,
                            colours=['#2c6fbb', '#e08214'],
                            title='The same three brain measurements, two different targets',
                            ylabel='AUROC')
plt.show()


🧠 **Think first:** The brain measurements predict *age* at least as well as they predict *impairment*. Does that make the model useless?

<details>
<summary>Click for one good answer</summary>

Not useless — but it does mean the model is not doing what its name suggests. Brain volume genuinely falls with age in everyone, and it falls faster with dementia. A model that has not been shown the person's age cannot tell those two causes apart, so a large part of its apparent 'diagnostic' skill is really age detection. That matters enormously in practice: a test that mostly detects age adds nothing to a clinician who can already see the patient's date of birth.

</details>


### 3.4 Simpson's paradox, on real data

The most counter-intuitive thing in this notebook. Look at the relationship between **education** and impairment across the whole cohort, and then inside each age band separately. The direction can reverse — because the older participants in this cohort had, on average, different educational opportunities from the younger ones, and age drives impairment.


In [ ]:
labelled = labelled.copy()
labelled['band'] = pd.cut(labelled['age'], [59, 70, 78, 100], labels=['60-70', '70-78', '78+'])

overall = labelled.groupby('education_code')['impaired'].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(overall.index, 100 * overall.values, 'o-', color='#c0392b', linewidth=2)
axes[0].set_title('Everyone pooled together')
axes[0].set_xlabel('education code (higher = more education)')
axes[0].set_ylabel('percent impaired')

for name, group in labelled.groupby('band', observed=True):
    rate = group.groupby('education_code')['impaired'].mean()
    axes[1].plot(rate.index, 100 * rate.values, 'o-', linewidth=2, label=f'age {name} (n={len(group)})')
axes[1].set_title('The same data, split by age band')
axes[1].set_xlabel('education code (higher = more education)')
axes[1].set_ylabel('percent impaired')
axes[1].legend(fontsize=9)
fig.suptitle("Simpson's paradox: the pooled trend need not be any group's trend", fontsize=12)
plt.tight_layout(); plt.show()

print(pd.crosstab(labelled['band'], labelled['education_code'], normalize='index').round(2))
print('\n^ The age bands do not have the same education profile. That is what causes the paradox.')


### 3.5 ✏️ Your turn — four ways to handle a confounder

Each of these is a real technique with real trade-offs. Switch between them and watch both the score *and* the sample size change.

- **`'ignore'`** — leave age in, pretend the problem does not exist.
- **`'exclude'`** — remove age from the features. Simple, but the *other* features still carry age.
- **`'stratify'`** — build a separate model inside each age band. Honest, but each model sees fewer people.
- **`'adjust'`** — regress age out of every brain measure first, then model the residuals. This is the standard epidemiological move.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try all four in turn. For each one, note BOTH the AUROC and the
#   number of people used. A higher score on fewer, more similar people
#   is not automatically a better result.
# ==========================================================================
STRATEGY = 'ignore'     # 'ignore', 'exclude', 'stratify' or 'adjust'

from sklearn.linear_model import LinearRegression

brain = ['nwbv', 'etiv_mm3', 'asf']
result_labels, result_scores, result_sizes = [], [], []

if STRATEGY == 'ignore':
    X_tr, X_te, y_tr, y_te = split_data(labelled[brain + ['age', 'sex']], y)
    result_labels = ['age left in']
    result_scores = [evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']]
    result_sizes = [len(labelled)]

elif STRATEGY == 'exclude':
    X_tr, X_te, y_tr, y_te = split_data(labelled[brain + ['sex']], y)
    result_labels = ['age removed']
    result_scores = [evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']]
    result_sizes = [len(labelled)]

elif STRATEGY == 'stratify':
    for name, group in labelled.groupby('band', observed=True):
        target = group['impaired'].astype(int)
        if target.nunique() < 2 or len(group) < 40:
            continue
        X_tr, X_te, y_tr, y_te = split_data(group[brain + ['sex']], target)
        result_labels.append(f'age {name}')
        result_scores.append(evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc'])
        result_sizes.append(len(group))

else:  # 'adjust'
    adjusted = labelled.copy()
    age_column = labelled[['age']].to_numpy()
    for measure in brain:
        expected = LinearRegression().fit(age_column, labelled[measure]).predict(age_column)
        adjusted[measure] = labelled[measure] - expected   # what is left after age is accounted for
    X_tr, X_te, y_tr, y_te = split_data(adjusted[brain + ['sex']], y)
    result_labels = ['age regressed out']
    result_scores = [evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']]
    result_sizes = [len(labelled)]

plots.plot_score_comparison([f'{a}\n(n={b})' for a, b in zip(result_labels, result_sizes)],
                            result_scores, reference=0.5,
                            colours=['#2c6fbb'] * len(result_scores),
                            title=f"Strategy: {STRATEGY}", ylabel='AUROC')
plt.show()
print('Remember: the goal is not the tallest bar. It is the bar you can defend.')


### 3.6 🔵 Your turn to write code — matching

The fifth technique, and the one closest to what a randomised trial does. **Matching** builds a subsample in which the impaired and unimpaired groups have the *same* age distribution, by pairing each impaired person with an unimpaired person of similar age and discarding everyone left over.

Fill in the `# TODO`. The scaffolding around it already works.


In [ ]:
cases = labelled[labelled.impaired == 1]
controls = labelled[labelled.impaired == 0]

# ✅ Worked solution: greedy nearest-neighbour matching on age, without replacement.
matched_rows = []
used = set()
for case_index, case in cases.iterrows():
    available = controls.drop(index=list(used), errors='ignore')
    if available.empty:
        break
    distance = (available['age'] - case['age']).abs()
    nearest = distance.idxmin()
    if distance[nearest] <= 3:            # only accept a close match
        used.add(nearest)
        matched_rows += [case_index, nearest]

# Why greedy-without-replacement: reusing one control for several cases would make the
# control group artificially homogeneous and understate its variance. Refusing matches
# worse than 3 years is the 'caliper' — it keeps the balance tight at the cost of throwing
# away cases with no comparable control, which is itself informative: if the oldest cases
# have no age-matched controls, then for those people the question is unanswerable in this
# cohort, and no amount of statistical adjustment can conjure the missing comparison.

matched = labelled.loc[matched_rows]
print(f'{len(matched)} people kept, {len(labelled) - len(matched)} discarded.')
print(matched.groupby('impaired')['age'].agg(['mean', 'std', 'count']).round(2))
plots.plot_by_group(matched, 'age', 'impaired',
                    title='After matching, the age distributions should sit on top of each other')
plt.show()
X_tr, X_te, y_tr, y_te = split_data(matched[['nwbv', 'etiv_mm3', 'asf', 'sex']],
                                    matched['impaired'].astype(int))
print(f"Matched-sample AUROC: {evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']:.3f}")


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** run 3.5 with all four strategies and write down, in one sentence each, what each one assumes.
- 🔵 **If you want to write code:** complete the matching TODO, then compare the matched-sample AUROC with the 'adjust' strategy's. Which do you trust more, and why?
- ⚫ **Take home:** propensity-score matching generalises 3.6 to many confounders at once. Look it up and consider what happens when the confounder you did not measure is the important one.


---
# 4 · Read the results

The deliverable of this module is not a score. It is a **judgement about what the score means**.


### 4.1 Every framing, side by side

One figure summarising the whole module.


In [ ]:
from sklearn.linear_model import LinearRegression

summary = {}

X_tr, X_te, y_tr, y_te = split_data(labelled[features], y)
summary['everything in'] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

X_tr, X_te, y_tr, y_te = split_data(labelled[['age']], y)
summary['age alone'] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

X_tr, X_te, y_tr, y_te = split_data(labelled[['nwbv', 'etiv_mm3', 'asf']], y)
summary['brain alone'] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

adjusted = labelled.copy()
age_column = labelled[['age']].to_numpy()
for measure in ['nwbv', 'etiv_mm3', 'asf']:
    adjusted[measure] = labelled[measure] - LinearRegression().fit(age_column, labelled[measure]).predict(age_column)
X_tr, X_te, y_tr, y_te = split_data(adjusted[['nwbv', 'etiv_mm3', 'asf']], y)
summary['brain, age removed'] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

narrow = labelled[(labelled.age >= 70) & (labelled.age <= 80)]
X_tr, X_te, y_tr, y_te = split_data(narrow[['nwbv', 'etiv_mm3', 'asf']], narrow['impaired'].astype(int))
summary['brain, ages 70-80 only'] = evaluate(train_model('logistic', X_tr, y_tr), X_te, y_te)['auroc']

plots.plot_score_comparison(list(summary), list(summary.values()), reference=0.5,
                            colours=['#e08214', '#c0392b', '#2c6fbb', '#5aa469', '#8e44ad'],
                            title='Five honest analyses of one dataset. They do not agree.',
                            ylabel='AUROC')
plt.show()
for name, value in summary.items():
    print(f'  {name:<26s} {value:.3f}')


### 4.2 Where the errors land

Subgroup analysis, which in this module is not a footnote but the finding.


In [ ]:
X_tr, X_te, y_tr, y_te = split_data(labelled[features], y)
final_model = train_model('logistic', X_tr, y_tr)
final_probability = final_model.predict_proba(X_te)[:, 1]
final_predicted = (final_probability >= 0.5).astype(int)

plots.plot_confusion(y_te, final_predicted, labels=('unimpaired', 'impaired'),
                     title='Everything-in model on held-out participants')
plt.show()

check = labelled.loc[X_te.index].copy()
check['correct'] = (final_predicted == y_te).astype(int)
check['band'] = pd.cut(check['age'], [59, 70, 78, 100], labels=['60-70', '70-78', '78+'])
for subgroup in ['band', 'sex', 'education_code']:
    plots.plot_subgroup_errors(check.dropna(subset=[subgroup]), subgroup, 'correct',
                               title=f'Proportion correct by {subgroup}')
    plt.show()


### 4.3 Shapley values — how much of the prediction *is* age?

Section 3.2 removed features one at a time and watched the score fall. **Shapley values** ask a sharper question, per person: how much did *this participant's* age, specifically, move *their* predicted risk?

The idea comes from game theory. Treat the features as players on a team and the prediction as the prize; a feature's Shapley value is its fair share, averaged over every order in which the team could have been assembled. With seven features that is all 2⁷ = 128 combinations, so we compute the **exact** values rather than the approximations the `shap` package uses.

Look at where `age` lands — and then compare it with the ablation figure in 3.2, which said something quite different.


In [ ]:
from interpret import shapley_values, shapley_importance, baseline_prediction

explain_columns = ['age', 'nwbv', 'etiv_mm3', 'asf', 'education_code', 'ses']
X_tr3, X_te3, y_tr3, y_te3 = split_data(labelled[explain_columns], y)
explain_model = train_model('random_forest', X_tr3, y_tr3)

shap_frame = shapley_values(explain_model, X_te3.head(60), X_tr3, features=explain_columns)
importance = shapley_importance(shap_frame)
plots.plot_importance(importance.index, importance.values,
                      title='Average influence on the predicted probability of impairment',
                      xlabel='mean |contribution| (exact Shapley values)')
plt.show()

share = importance / importance.sum()
for name, value in share.items():
    print(f'  {name:<16s} {value:.0%} of all the movement in this model\'s predictions')
print()
print(f"Age accounts for {share['age']:.0%} of it — yet section 3.2 showed that DELETING the age")
print('column costs the model almost nothing. Hold both of those facts in your head, then')
print('open the box below.')


🧠 **Think first:** Ablation said deleting `age` costs roughly nothing. Shapley says age drives about a quarter of every prediction. Which is wrong?

<details>
<summary>Click for one good answer</summary>

Neither. They answer different questions, and the gap between them is the single most useful thing in this module.

**Ablation** asks *"what happens to the score if I delete this column?"* Almost nothing happens — because `nwbv`, `asf` and `etiv_mm3` are themselves correlated with age, so when you remove the age column the model simply reads age off the brain measurements instead. Ablation therefore reports "unimportant" for a confounder whenever *anything else* can stand in for it.

**Shapley values** ask *"among the features actually present, how is the credit for this prediction divided?"* Age is genuinely doing about a quarter of the work, and it says so.

The practical lesson, and it is a trap people fall into constantly: **a feature you can delete without losing accuracy is not a feature the model was ignoring.** To find out whether age matters you have to break its link with the outcome — stratify, adjust or match, as in 3.5 and 3.6 — not merely drop the column and watch the score.

</details>


### 4.4 ✏️ Your turn — one participant at a time

Pick individual people and read what the model based their prediction on. Look for somebody whose risk was driven almost entirely by their age.


In [ ]:
# ==========================================================================
# ✏️  YOUR TURN
#   Try several values of PERSON (0 to 59).
#   For each, read the bars as a sentence: 'their predicted risk was
#   pushed up mainly by ___ and pulled down by ___'.
#   Then ask: would a clinician have needed a model to say that?
# ==========================================================================
PERSON = 0

contributions = shap_frame.iloc[PERSON].sort_values()
plots.plot_importance(contributions.index, contributions.values,
                      title=f'Participant {X_te3.index[PERSON]}: what drove their predicted risk',
                      xlabel='contribution to predicted probability (blue raises risk)')
plt.show()

print('Their actual measurements:')
print(X_te3.iloc[PERSON].round(3).to_string())
average = baseline_prediction(explain_model, X_tr3)
print()
print(f'  cohort average risk   {average:.3f}')
print(f'  + contributions       {contributions.sum():+.3f}')
print(f'  = predicted risk      {average + contributions.sum():.3f}')
print(f'  actually impaired?    {"yes" if y_te3.iloc[PERSON] == 1 else "no"}')


### 4.5 What this module is for

You built a model with a respectable AUROC and then showed that a large part of it was **age**, and that the apparent effect of education **reverses** depending on how you slice the data. Neither finding is a bug in the code. Both are properties of how the cohort was recruited.

Three things worth carrying into every other module today:

1. **A confounder is not noise.** It is a real cause of both the feature and the outcome. You cannot average it away; you have to design around it.
2. **A high score is a question, not an answer.** The right response to 'my model got 0.92' is '0.92 of what, measured how, on whom?'
3. **The choice of analysis is a scientific claim.** Adjusting for age, or not, is an assertion about what you think is causing what. Make it explicitly, and say so in the write-up.

**On bias.** OASIS-1 is volunteers from one American city — predominantly white, predominantly educated, healthy enough to lie still in a scanner. `ses` is missing most often for the people whose socioeconomic circumstances are hardest to record. Every confounding problem in this notebook exists in a harsher form for groups who are underrepresented, because there are fewer of them to adjust with.

---

### 🧠 Final question for the group discussion

Somebody shows you a dementia-screening model with an AUROC of 0.94, trained on a hospital's memory clinic records. **What three questions do you ask before believing it?**


### 🎚 Go further — pick whichever suits you

- 🟢 **Everyone:** pick which of the five bars in 4.1 you would put in a paper's abstract, and be ready to defend it.
- 🔵 **If you want to write code:** add `mmse` to the everything-in model and watch the AUROC jump. Then explain why that is the least interesting model in the notebook.
- ⚫ **Take home:** read one paper that reports a dementia-prediction AUROC and find out how it handled age. Many do not say.
